# PaddleOCR-VL: PDF → paddle.json + paddle_blocks.json

Reads papers with PaddleOCR-VL and writes, to `thesis/papers/<PAPER_ID>/` on Drive:
- `paddle.json`: every display formula as LaTeX, with its equation number and page
- `paddle_blocks.json`: every block on every page in reading order (text, titles, tables as HTML)

Copy both files into `extractor_agent/papers/<PAPER_ID>/` and run the extractor locally.

**Runtime:** GPU (T4).

**One paper or a batch** (Settings):
- `BATCH = []`: only `PAPER_ID`.
- `BATCH = ['owhondah_2016', 'nagy_2008']`: those papers, one after the other.
- `BATCH = 'new'`: every `thesis/papers/<id>.pdf` that has no `paddle_blocks.json` yet.

Upload each PDF as `thesis/papers/<id>.pdf`. A supplementary file `thesis/papers/<id>_supp.pdf` is read too when it exists. A paper that fails is reported and the batch goes on. With `SKIP_DONE = True` a paper that already has its output is skipped, so after a disconnect just run again.

**Order:** Settings, mount, A1 install, restart, Settings, mount, check, then A2 and A3. After the first install in a session, only A2 and A3 are needed for more papers.

## 1. Settings

In [ ]:
PAPER_ID  = 'cheema_2018'          # used when BATCH is empty
BATCH     = []                     # [] = PAPER_ID only | ['id1', 'id2'] | 'new' = every PDF without output yet
SKIP_DONE = True                   # skip papers whose paddle_blocks.json already exists (batch only)

THESIS   = '/content/drive/MyDrive/thesis'
PAPERS   = f'{THESIS}/papers'      # PDFs: papers/<id>.pdf, optional papers/<id>_supp.pdf; output: papers/<id>/
PAGES    = None     # 0-indexed pages to read; None = all pages (needed for the text and tables). Single paper only.
DPI      = 150      # VLMs are trained on ~1-2k px images; 300 DPI costs memory for no gain

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs(PAPERS, exist_ok=True)
# Paddle's GPU build needs the NVIDIA driver (libcuda.so.1), which exists only on a GPU runtime.
# "ImportError: libcuda.so.1" when importing paddle means this session has no GPU: either the runtime
# type is CPU, or Colab's GPU limit was reached and it reconnected you on a CPU machine.
import ctypes
try:
    ctypes.CDLL('libcuda.so.1')
except OSError:
    raise SystemExit('No GPU in this session. Runtime > Change runtime type > T4 GPU, then run again. '
                     'If no GPU is offered, the free GPU quota is used up: wait a few hours or use Colab Pro.')
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

---
## Pipeline

### A1. Install

PaddlePaddle ships CUDA-specific wheels from its own index. `cu126` is the build the PaddleOCR-VL docs show; Colab's driver is newer, which is fine (drivers are backward compatible). If install or the check below fails, try `PADDLE_VER = '3.3.1'` or the `cu129` index.

**Speed-ups kept on Drive** (`thesis/wheels/`, `thesis/paddlex_models/`): the first session downloads the Paddle wheel (~2 GB) and the OCR models (~2 GB) and saves them to Drive; later sessions copy them from Drive, which is much faster. Delete those folders to force a fresh download (e.g. after changing `PADDLE_VER`).


In [ ]:
PADDLE_VER = '3.2.1'
# Paddle's download server is slow from Colab (~2 GB wheel). The wheel is kept on Drive after the
# first session, so later sessions install it from Drive instead of downloading it again.
import os, glob
WHEELS = f'{THESIS}/wheels/paddlepaddle-gpu-{PADDLE_VER}-cu126'
os.makedirs(WHEELS, exist_ok=True)
if not glob.glob(f'{WHEELS}/paddlepaddle_gpu-*.whl'):
    print('first time: downloading the Paddle wheel to Drive (slow, once)')
    !pip download -q --no-deps --timeout 600 --retries 10 -d "{WHEELS}" paddlepaddle-gpu=={PADDLE_VER} -i https://www.paddlepaddle.org.cn/packages/stable/cu126/
else:
    print('Paddle wheel found on Drive:', os.path.basename(glob.glob(f'{WHEELS}/paddlepaddle_gpu-*.whl')[0]))
!pip install -q --find-links "{WHEELS}" paddlepaddle-gpu=={PADDLE_VER}
!pip install -q -U "paddleocr[doc-parser]" pymupdf

# paddlepaddle-gpu pulls in an older NCCL that breaks Colab's torch
# ('undefined symbol: ncclCommWindowDeregister'), and paddleocr imports torch if present.
# Put back the NCCL version torch was built for. Newer NCCL stays compatible with paddle.
from importlib.metadata import requires
nccl = [r.split(';')[0].strip() for r in requires('torch') if r.startswith('nvidia-nccl')]
print('restoring', nccl)
!pip install -q --force-reinstall --no-deps "{nccl[0]}"

# Stop here if the install did not finish: a failed download does not stop the cell.
!pip show paddlepaddle-gpu paddleocr | grep -E '^(Name|Version)'


**Restart the session now** (Runtime → Restart session), then run the Settings and mount cells again and continue from the check below. Do not reinstall.

The install replaces packages Colab already loaded (numpy, protobuf, ...). Importing `paddleocr` without a restart fails, and any second attempt then fails with `PDX has already been initialized`, which hides the real error. If `from paddleocr import PaddleOCRVL` fails after a clean restart, the first error message is the one that matters.

In [ ]:
import paddle
paddle.utils.run_check()          # must end with 'PaddlePaddle is installed successfully!'
print(paddle.__version__, paddle.device.get_device())

### A2. The OCR steps (run once per session)

Defines `extract_paper(paper_id)`: render the pages, run PaddleOCR-VL, collect the formulas into `paddle.json` and every block into `paddle_blocks.json`, then the supplementary file if there is one. `paddle_blocks.json` is written last, so its presence means the paper finished.

Formula labels: any label containing `formula` is kept (inline math excluded); the printed label counts show if equations come under another label.
Equation numbers: from a `\tag{n}` or trailing `(n)` in the LaTeX; otherwise the number block that follows in reading order (right when the number sits below a wide formula, and on two-column pages); otherwise a number on the same line, to the right, nearest first.

In [ ]:
import os, json, glob, re, shutil, time, collections, pymupdf

def eq_number(text):                  # main paper: (12) -> 12
    m = re.search(r'\\tag\{(\d+)\}', text) or re.search(r'\((\d+)\)\s*\$*\s*$', text)
    return int(m.group(1)) if m else None

def supp_number(text):                # supplementary: (A-3), (S1), (12a) kept as text
    m = re.search(r'\\tag\{([^}]+)\}', text) or re.search(r'\(([A-Z]{0,2}\s*[-.]?\s*\d+[a-z]?)\)\s*\$*\s*$', text)
    return re.sub(r'\s', '', m.group(1)) if m else None

def y_overlap(a, b):
    return min(a[3], b[3]) - max(a[1], b[1]) > 0.5 * min(a[3] - a[1], b[3] - b[1])

def get_pipeline():
    """Loaded once per session; the OCR models (~2 GB) are kept on Drive and copied back in later sessions."""
    global pipeline
    if 'pipeline' in globals():
        return pipeline
    from paddleocr import PaddleOCRVL
    local, saved = os.path.expanduser('~/.paddlex'), f'{THESIS}/paddlex_models'
    if os.path.isdir(saved):
        print('copying the OCR models from Drive ...')
        shutil.copytree(saved, local, dirs_exist_ok=True)
    pipeline = PaddleOCRVL()          # default pipeline version, PaddlePaddle engine
    if os.path.isdir(local) and not os.path.isdir(saved):
        print('saving the OCR models to Drive for next time ...')
        shutil.copytree(local, saved)
    return pipeline

def ocr(pdf_path, tag, pages=None):
    """Render and read every page; returns [(page, blocks)]. tag keeps main and supplementary apart."""
    img_dir, raw_dir = f'/content/pages_{tag}', f'/content/raw_{tag}'
    shutil.rmtree(img_dir, ignore_errors=True); shutil.rmtree(raw_dir, ignore_errors=True)   # never mix papers
    os.makedirs(img_dir)
    doc = pymupdf.open(pdf_path)
    out = []
    todo_pages = list(pages if pages is not None else range(doc.page_count))
    for i, p in enumerate(todo_pages, 1):
        t0 = time.time()
        path = f'{img_dir}/p{p}.png'
        doc[p].get_pixmap(dpi=DPI).save(path)
        for res in get_pipeline().predict(path):
            res.save_to_json(save_path=f'{raw_dir}/p{p}')
        f = glob.glob(f'{raw_dir}/p{p}/**/*.json', recursive=True)[0]
        out.append((p, json.load(open(f))['parsing_res_list']))
        print(f'    {tag} page {i}/{len(todo_pages)}: {len(out[-1][1])} blocks, {time.time() - t0:.0f} s', flush=True)
    return out

def formulas_of(pages, number, source):
    formulas, labels = [], collections.Counter()
    for p, blocks in pages:
        labels.update(b['block_label'] for b in blocks)
        numbers = [b for b in blocks if 'number' in b['block_label']]
        for k, b in enumerate(blocks):
            lab = b['block_label']
            if 'formula' not in lab or 'number' in lab or 'inline' in lab:   # inline math inside a sentence is not an equation
                continue
            tex, box = b['block_content'], list(map(float, b['block_bbox']))
            n = number(tex)
            if n is None:
                if k + 1 < len(blocks) and 'number' in blocks[k + 1]['block_label']:
                    near = [(0, blocks[k + 1])]
                else:
                    near = sorted(((list(map(float, x['block_bbox']))[0] - box[2], x) for x in numbers
                                   if y_overlap(box, list(map(float, x['block_bbox'])))
                                   and list(map(float, x['block_bbox']))[0] - box[2] > -10), key=lambda t: t[0])
                if near and source == 'main':         # as before: the digits of the number block ('(C-5)' -> 5)
                    m = re.search(r'\d+', near[0][1]['block_content'])
                    n = int(m.group()) if m else None
                elif near:                            # supplementary: kept as text ('(S-3)' -> 'S-3')
                    n = supp_number(near[0][1]['block_content'])
            formulas.append({'eq': n, 'page': p, 'label': lab, 'latex': tex, 'bbox': box, 'route': 'pipeline', 'source': source})
    return formulas, labels

def blocks_of(pages, source):
    return [{'page': p, 'order': k, 'label': b['block_label'], 'content': b['block_content'],
             'bbox': list(map(float, b['block_bbox'])), 'source': source} for p, blocks in pages for k, b in enumerate(blocks)]

def extract_paper(paper_id, pages=None):
    pdf, supp, out_dir = f'{PAPERS}/{paper_id}.pdf', f'{PAPERS}/{paper_id}_supp.pdf', f'{PAPERS}/{paper_id}'
    if not os.path.exists(pdf):
        raise FileNotFoundError(f'no PDF at {pdf}')
    os.makedirs(out_dir, exist_ok=True)
    main = ocr(pdf, 'main', pages)
    formulas, labels = formulas_of(main, eq_number, 'main')
    blocks = blocks_of(main, 'main')
    n_supp = 0
    if os.path.exists(supp):
        sp = ocr(supp, 'supp')
        f2, l2 = formulas_of(sp, supp_number, 'supplementary')
        formulas += f2; labels += l2; blocks += blocks_of(sp, 'supplementary'); n_supp = len(f2)
    nums = [f['eq'] for f in formulas if f['source'] == 'main' and f['eq'] is not None]
    bad = [(a, b) for a, b in zip(nums, nums[1:]) if b <= a]
    head = {'paper': paper_id, 'reader': 'PaddleOCR-VL pipeline', 'dpi': DPI}
    json.dump({**head, 'formulas': formulas}, open(f'{out_dir}/paddle.json', 'w'), indent=1, ensure_ascii=False)
    json.dump({**head, 'blocks': blocks}, open(f'{out_dir}/paddle_blocks.json', 'w'), indent=1, ensure_ascii=False)   # last: marks done
    return {'paper': paper_id, 'pages': len(main), 'formulas': len(formulas), 'supplementary_formulas': n_supp,
            'unnumbered': sum(f['eq'] is None for f in formulas), 'numbering': 'ok' if not bad else f'not increasing at {bad}',
            'blocks': len(blocks), 'labels': dict(labels)}
print('ready')

### A3. Run

First run downloads the layout and VL models (~2 GB). If it runs out of GPU memory: lower `DPI` to 120. A summary of every paper is saved to `thesis/papers/ocr_log.json`: check `numbering` (equation numbers should increase in reading order) and `unnumbered` for each paper before running the extractor.

In [ ]:
import time, traceback
if BATCH == 'new':
    todo = sorted(os.path.basename(f)[:-4] for f in glob.glob(f'{PAPERS}/*.pdf')
                  if not f.endswith('_supp.pdf') and not os.path.exists(f'{f[:-4]}/paddle_blocks.json'))
elif BATCH:
    todo = [p for p in BATCH if not (SKIP_DONE and os.path.exists(f'{PAPERS}/{p}/paddle_blocks.json'))]
    skipped = [p for p in BATCH if p not in todo]
    if skipped: print('already done, skipped:', skipped)
else:
    todo = [PAPER_ID]
print('to read:', todo)

log_file = f'{PAPERS}/ocr_log.json'
log = json.load(open(log_file)) if os.path.exists(log_file) else {}
for i, pid in enumerate(todo, 1):
    t = time.time()
    print(f'\n[{i}/{len(todo)}] {pid} ...')
    try:
        r = extract_paper(pid, PAGES if not BATCH else None)
        r['minutes'] = round((time.time() - t) / 60, 1)
        print(f"  {r['pages']} pages, {r['formulas']} formulas ({r['supplementary_formulas']} supplementary), "
              f"{r['unnumbered']} without a number, numbering {r['numbering']}, {r['minutes']} min")
    except Exception as e:
        r = {'paper': pid, 'error': f'{type(e).__name__}: {e}'}
        print('  FAILED:', r['error']); traceback.print_exc()
    log[pid] = {**r, 'when': time.strftime('%Y-%m-%d %H:%M')}
    json.dump(log, open(log_file, 'w'), indent=1)          # after every paper, so a disconnect loses nothing

print('\nsummary:')
for pid in todo:
    r = log[pid]
    print(f"  {pid:28} " + (f"FAILED {r['error']}" if 'error' in r else
          f"{r['formulas']:3} formulas, {r['unnumbered']} unnumbered, numbering {r['numbering']}"))

### A4. Look at one paper's formulas (optional)

Prints every formula with its number, to compare against the PDF.

In [ ]:
SHOW = todo[0] if todo else PAPER_ID
for f in json.load(open(f'{PAPERS}/{SHOW}/paddle.json'))['formulas']:
    print(f"{str(f['eq']):>6}  p.{f['page'] + 1}  {f['latex'][:150]}")